# LatentTokenGuard caption benchmark: LLaVA-1.5-7B run

Runs the object-level hallucination benchmark on a cloud GPU.

**Kaggle:** Settings → Accelerator **GPU T4** · Internet **On**.
**Colab:** Runtime → GPU (T4 or A100); run the Drive cell so outputs survive disconnects.

The run is resumable: re-running the run cell skips images already captioned in `OUT`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Settings
`MODEL`: `llava15_7b_4bit` on a 16 GB T4, `llava15_7b` (fp16) on an A100.

In [ ]:
import os
MODEL = "llava15_7b_4bit"      # or "llava15_7b" on A100
N_IMAGES = 1000                # fixed sample (seed 42), the same on every session
LIMIT = None                   # e.g. 400 to stop after 400 new images in this session
REPO = "https://github.com/sunilsk17/LatentTokenGuard.git"

ON_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/ltg_runs/{MODEL}_n{N_IMAGES}_s42"
    WORK = "/content"
else:  # Kaggle: /kaggle/working is saved with the notebook version
    OUT = f"/kaggle/working/{MODEL}_n{N_IMAGES}_s42"
    WORK = "/kaggle/working"
COCO = "/tmp/coco"
print("outputs →", OUT)

## 2. Code and dependencies

In [ ]:
%cd {WORK}
!rm -rf LatentTokenGuard && git clone -q {REPO} LatentTokenGuard
%cd LatentTokenGuard
!git log -1 --oneline
# Do not install requirements.txt here: it pins transformers 4.43 for the old POPE code.
!pip install -q -U bitsandbytes sentencepiece
import transformers, torch; print(transformers.__version__, torch.__version__)

## 3. COCO val2017 (≈1 GB, into /tmp so it is not saved as output)

In [ ]:
import os
if not os.path.exists(f"{COCO}/val2017"):
    !mkdir -p {COCO}
    !wget -q http://images.cocodataset.org/zips/val2017.zip -O {COCO}/val2017.zip
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O {COCO}/ann.zip
    !cd {COCO} && unzip -q val2017.zip && unzip -q ann.zip && rm val2017.zip ann.zip
!ls {COCO}/val2017 | wc -l; ls {COCO}/annotations

## 4. Quick timing check (20 images)
Uses the same output folder, so these 20 count towards the run.

In [ ]:
!python experiments/caption_run.py --model {MODEL} --n_images {N_IMAGES} --limit 20 \
    --images_dir {COCO}/val2017 --annotations_dir {COCO}/annotations --out {OUT}
import json, numpy as np
recs = [json.loads(l) for l in open(f"{OUT}/captions.jsonl")]
gen = np.mean([r["gen_seconds"] + r["feature_seconds"] for r in recs])
print(f"{len(recs)} images, {gen:.1f} s/image → ~{gen * (N_IMAGES - len(recs)) / 3600:.1f} h for the rest")
print("mean tokens:", np.mean([r["n_tokens"] for r in recs]),
      "| mentions/image:", np.mean([len(r["mentions"]) for r in recs]),
      "| hallucinated share:", np.mean([m["label"] for r in recs for m in r["mentions"]] or [0]))
print(recs[0]["caption"][:400])

## 5. Full run (resumable)

In [ ]:
limit = f"--limit {LIMIT}" if LIMIT else ""
!python experiments/caption_run.py --model {MODEL} --n_images {N_IMAGES} {limit} \
    --images_dir {COCO}/val2017 --annotations_dir {COCO}/annotations --out {OUT}
!wc -l {OUT}/captions.jsonl

## 6. Evaluate

In [ ]:
!python experiments/caption_eval.py --run {OUT}
from IPython.display import Image, display
display(Image(f"{OUT}/eval/layer_map.png")); display(Image(f"{OUT}/eval/roc_test.png"))

## 7. Save the run
Kaggle: **Save Version → Save & Run All** keeps `/kaggle/working`. To resume in a new session, attach that
version's output as an input dataset and copy the run folder back to `OUT` before step 5.
Colab: the run is already on Drive.

Zip it to download for local evaluation:

In [ ]:
!cd {os.path.dirname(OUT)} && zip -qr {os.path.basename(OUT)}.zip {os.path.basename(OUT)} && ls -lh {os.path.basename(OUT)}.zip